# Parsing Propositional Logic

F# counterpart of `Python/Chapter-10/Propositional-Logic.ipynb`. Build once with `python FSharp/tools/build.py`, then run this notebook from top to bottom with a fresh .NET (F#) kernel. The scanner and parser are generated from real FsLexYacc specifications.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "Lalr/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "Lalr/bin/Debug/net9.0/Lalr.dll"
#load "../Support/Visualization.fsx"
open System
open System.IO
open FormalLanguages.Lalr
open Visualization

## Typed syntax and scanner

The semantic actions construct a closed expression type. Numeric tokens are converted once, in the scanner; arithmetic entry points require integer literals, and the symbolic calculator accepts decimal and exponent forms; punctuation becomes a token constructor. Newlines advance the source position, and illegal input raises a positioned error.

```fsharp
namespace FormalLanguages.Lalr

/// A closed syntax tree makes every interpreter case visible to the compiler.
type Expression =
    | Number of float
    | Name of string
    | Unary of string * Expression
    | Binary of string * Expression * Expression

type Statement =
    | Assign of string * Expression
    | Evaluate of Expression

type Lexeme =
    | Numeric of float
    | Identifier of string
    | Operator of string
    | Character of char
    | End
```

The shared `.fsl` file supplies an expression mode and a character mode:

```text
{
module FormalLanguages.Lalr.Lexer
open System.Globalization
open FSharp.Text.Lexing
}
let digit = ['0'-'9']
let letter = ['a'-'z' 'A'-'Z']
let number = ('0' | ['1'-'9'] digit*) ('.' digit+)? (['e' 'E'] ['+' '-']? digit+)?
rule token = parse
| [' ' '\t' '\r']+ { token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
| number { Numeric (System.Double.Parse(LexBuffer<char>.LexemeString lexbuf, CultureInfo.InvariantCulture)) }
| letter (letter | digit | ['_' '<' '>'])* { Identifier (LexBuffer<char>.LexemeString lexbuf) }
| ":=" { Operator ":=" }
| ['+' '-' '*' '/' '^' '(' ')' ';' '↔' '→' '∧' '∨' '¬' '⊤' '⊥'] { Operator (LexBuffer<char>.LexemeString lexbuf) }
| eof { End }
| _ { failwithf "Illegal character %A at line %d, column %d" (LexBuffer<char>.LexemeString lexbuf) lexbuf.StartPos.Line (lexbuf.StartPos.Column + 1) }
and character = parse
| [' ' '\t' '\r']+ { character lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; character lexbuf }
| ['U' 'V' 'W' 'X' 'v' 'w' 'x' 'y' 'z'] { Character (lexbuf.LexemeChar 0) }
| eof { End }
| _ { failwithf "Illegal character %A" (LexBuffer<char>.LexemeString lexbuf) }
```

## Parser specification

FsYacc generates token constructors and parse tables from this `.fsy` specification. `$1`, `$2`, and `$3` refer to values on the right-hand side. `main` consumes `EOF`, so a valid prefix with trailing junk is rejected.

```text
%{
open FormalLanguages.Lalr
%}
%token <string> ID
%token EQUIV IMPLIES OR AND NOT TRUE FALSE LPAREN RPAREN EOF
%nonassoc EQUIV
%right IMPLIES
%left OR
%left AND
%right NOT
%start main
%type <FormalLanguages.Lalr.Expression> main
%%
main: formula EOF { $1 }
formula:
  | implication EQUIV implication { Binary ("↔", $1, $3) }
  | implication { $1 }
implication:
  | disjunction IMPLIES implication { Binary ("→", $1, $3) }
  | disjunction { $1 }
disjunction:
  | disjunction OR conjunction { Binary ("∨", $1, $3) }
  | conjunction { $1 }
conjunction:
  | conjunction AND negation { Binary ("∧", $1, $3) }
  | negation { $1 }
negation:
  | NOT negation { Unary ("¬", $2) }
  | TRUE { Name "⊤" }
  | FALSE { Name "⊥" }
  | ID { Name $1 }
  | LPAREN formula RPAREN { $2 }
```

In [ ]:
printfn "%s" (File.ReadAllText "Lalr/Logic.fsyacc.output")

In [ ]:
let rec toTree expression =
    match expression with
    | Number value -> Leaf (string value)
    | Name name -> Leaf name
    | Unary (operator, operand) -> Node (operator, [toTree operand])
    | Binary (operator, left, right) -> Node (operator, [toTree left; toTree right])

let test parse text =
    let expression = parse text
    printfn "%s ⇒ %A" text expression
    expression

Equivalence is non-associative: its two operands are implications. This grammar encodes that constraint structurally, avoiding the pinned generator’s treatment of equal `%nonassoc` precedence. Implication associates to the right, disjunction and conjunction to the left, and negation binds most tightly.

In [ ]:
let examples =
    [ "(p ↔ q) ∨ (q ↔ p) → (¬q ↔ ¬p) ∧ (q → ⊥ ↔ p → ⊥)"
      "p ∧ ⊤"; "q ∨ ⊥"; "¬p ∧ q"; "p ∧ q → r"; "p → q → r"
      "¬(p ∨ q) ↔ ¬p ∧ ¬q"; "¬¬p ↔ p"
      "(p → q) ∧ (q → r) → p → r"; "p ∨ q ∧ r"
      "v ↔ u → p ∨ q ∧ ¬ ⊥" ]
let trees = examples |> List.map (test Parse.logic)
checkNotebook (Parse.logic "p → q → r" = Binary ("→", Name "p", Binary ("→", Name "q", Name "r")))
checkNotebook (Parse.logic "p ∨ q ∧ r" = Binary ("∨", Name "p", Binary ("∧", Name "q", Name "r")))
let nonAssociative = try Parse.logic "p ↔ q ↔ r" |> ignore; false with _ -> true
checkNotebook nonAssociative

## Evaluate a formula

A valuation maps proposition names to truth values. The compiler checks that every syntax-tree constructor is handled. Missing propositions raise an explicit error.

In [ ]:
let rec evaluate values formula =
    match formula with
    | Name "⊤" -> true
    | Name "⊥" -> false
    | Name name -> Map.find name values
    | Unary ("¬", operand) -> not (evaluate values operand)
    | Binary (operator, left, right) ->
        let p, q = evaluate values left, evaluate values right
        match operator with
        | "∧" -> p && q
        | "∨" -> p || q
        | "→" -> not p || q
        | "↔" -> p = q
        | _ -> invalidArg "formula" operator
    | other -> invalidArg "formula" (sprintf "Not a logical formula: %A" other)

let tautology = Parse.logic "¬(p ∨ q) ↔ ¬p ∧ ¬q"
for p in [false; true] do
    for q in [false; true] do
        checkNotebook (evaluate (Map.ofList ["p", p; "q", q]) tautology)

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(trees[6] |> toTree |> treeToDot |> renderDot), "text/html") |> ignore